# NeuMF — chạy pipeline trên Google Colab

Code lấy từ GitHub, dữ liệu đọc từ Google Drive, chạy `run.py all` trên H&M (mẫu `hm_subset` ~300k dòng theo khách, trải 2 năm — đúng cấu hình trong báo cáo).
Kết quả lưu vào `My Drive/result/<thời điểm chạy>/`.

**Trước khi chạy:**
- **Commit + push code mới nhất lên nhánh `main`** — Colab clone từ GitHub, thay đổi chưa push sẽ không có trên Colab.
- `Runtime → Change runtime type → GPU (T4)`: pipeline tự dùng GPU nếu có (`device: auto`), không thì chạy CPU (chậm hơn nhiều).
- Upload `transactions_train.csv` (H&M) lên Drive vào thư mục `DATA_DIR` (mặc định `My Drive/data`).
- Repo private: thêm secret `GITHUB_TOKEN` (biểu tượng 🔑 bên trái) và bật quyền cho notebook.

In [ ]:
!nvidia-smi -L || echo "Không có GPU — sẽ chạy trên CPU"

## 1. Cấu hình, mount Drive, clone repo

In [ ]:
import os, time
from google.colab import drive, userdata

REPO_URL = "https://github.com/sulinh3625/DU-AN-CNTT.git"
BRANCH = "main"
DATASET = "hm_subset"                        # hoặc "hm" (quy mô đầy đủ, cần cache Parquet)
STAMP = time.strftime("%Y%m%d_%H%M")
DATA_DIR = "/content/drive/MyDrive/data"      # nơi đặt CSV gốc trên Drive
RESULT_DIR = f"/content/drive/MyDrive/result/{STAMP}"

drive.mount("/content/drive")
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
CLONE_URL = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
print("Dữ liệu:", DATA_DIR)
print("Kết quả sẽ lưu vào:", RESULT_DIR)

In [ ]:
%cd /content
!rm -rf DU-AN-CNTT
!git clone -q --depth 1 --branch $BRANCH $CLONE_URL DU-AN-CNTT
%cd /content/DU-AN-CNTT/neumf_project_v2
!git log --oneline -1
!pip install -q -r requirements.txt

## 2. Liên kết dữ liệu từ Drive vào `data/raw/hm/`

Dùng symlink, không chép file (3,5 GB). Với `hm_subset`, mẫu ~300k dòng (`scripts/00_sample_hm300k.py`) được tạo **một lần** vào `DATA_DIR/hm300k_transactions.csv` rồi dùng lại cho các phiên sau.
Với `hm`, cache Parquet được tạo **một lần** vào `DATA_DIR/hm_cache` trên Drive rồi dùng lại cho các phiên sau.

In [ ]:
import shutil
from pathlib import Path

src, dst = Path(DATA_DIR) / "transactions_train.csv", Path("data/raw/hm/transactions_train.csv")
if not src.exists():
    raise FileNotFoundError(f"Thiếu {src} — upload file lên Drive hoặc sửa DATA_DIR")
dst.parent.mkdir(parents=True, exist_ok=True)
dst.unlink(missing_ok=True)
dst.symlink_to(src)
print(f"{dst} -> {src} ({src.stat().st_size / 1e6:,.0f} MB)")

if DATASET == "hm_subset":  # configs/hm_subset.yaml đọc data/processed/hm/hm300k_transactions.csv
    sample = Path(DATA_DIR) / "hm300k_transactions.csv"
    if not sample.exists():
        !python scripts/00_sample_hm300k.py --out-path "{sample}"
    link = Path("data/processed/hm/hm300k_transactions.csv")
    link.parent.mkdir(parents=True, exist_ok=True)
    link.unlink(missing_ok=True)
    link.symlink_to(sample)
    print(f"{link} -> {sample}")

if DATASET == "hm":  # configs/hm.yaml đọc data/processed/hm/transactions_clean.parquet
    cache = Path(DATA_DIR) / "hm_cache"
    if not (cache / "transactions_clean.parquet").exists():
        !python scripts/00_prepare_hm_cache.py --out-dir "{cache}"
    proc = Path("data/processed/hm")
    if proc.is_symlink():
        proc.unlink()
    elif proc.exists():
        shutil.rmtree(proc)
    proc.symlink_to(cache, target_is_directory=True)
    print(f"{proc} -> {cache}")

## 3. Kiểm tra unit test

In [ ]:
!python -m pytest -q 2>&1 | tail -3

## 4. Chạy pipeline

`run.py all` = audit → GMF → MLP → EarlyFusion → NeuMF-Scratch → NeuMF-Pretrained → baselines → bảng + biểu đồ.
Log được ghi thẳng vào `RESULT_DIR` nên vẫn còn nếu Colab ngắt giữa chừng.

In [ ]:
os.makedirs(RESULT_DIR, exist_ok=True)
TAG = f"hm_colab_{STAMP}"                    # tiền tố hm_ để demo tự nhận run mới nhất
!python -u run.py all --config configs/$DATASET.yaml --run-tag $TAG 2>&1 | tee "$RESULT_DIR/run.log"
# tee nuốt mã lỗi của python -> tự kiểm tra đầu ra
if not Path(f"outputs/tables/{TAG}").exists():
    raise RuntimeError(f"Pipeline lỗi — xem {RESULT_DIR}/run.log")

## 5. Lưu kết quả vào My Drive/result

Chép `outputs/` (experiments, tables, figures, data_audit) — bỏ checkpoint `*.pt` và thư mục `archive` có sẵn trong repo.

In [ ]:
import shutil
shutil.copytree("outputs", RESULT_DIR, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("archive", "checkpoints", ".gitkeep", "*.pt"))
!find "$RESULT_DIR" -type f | sort

## 6. Xem nhanh kết quả

In [ ]:
import pandas as pd
from IPython.display import display, Image

for f in sorted(Path(f"outputs/tables/{TAG}").glob("*.csv")):
    print(f.name); display(pd.read_csv(f).round(5))
for f in sorted(Path(f"outputs/figures/{TAG}").glob("*.png")):
    print(f.name); display(Image(str(f), width=800))